In [ ]:
import pandas as pd
import torch
import numpy as np
import time
import os
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

class Config:
    train_path = "/home/aman_swaraj/Downloads/Codelite/train_original.csv"
    test_path = "/home/aman_swaraj/Downloads/Codelite/test_original.csv"
    tag_vocab_path = "/home/aman_swaraj/Downloads/Codelite/unique_tags.csv"
    results_dir = "/home/aman_swaraj/Downloads/Codelite/ast_experiment_results"
    models_to_test = ["ngram_tfidf_svm", "bow_logistic", "handcrafted_lgb"]
    seed = 42

class CodeFeatureExtractor:
    @staticmethod
    def extract_features(code_string):
        features = {}
        features['length'] = len(code_string)
        features['num_lines'] = code_string.count('\n') + 1
        features['avg_line_length'] = len(code_string) / max(1, features['num_lines'])
        keywords = ['def', 'class', 'import', 'from', 'if', 'else', 'for', 'while', 'return', 'try', 'except']
        for kw in keywords:
            features[f'kw_{kw}'] = code_string.count(kw)
        features['num_indents'] = code_string.count('    ') + code_string.count('\t')
        features['num_parentheses'] = code_string.count('(') + code_string.count(')')
        features['num_brackets'] = code_string.count('[') + code_string.count(']')
        features['num_braces'] = code_string.count('{') + code_string.count('}')
        operators = ['=', '+', '-', '*', '/', '%', '==', '!=', '<', '>', '<=', '>=', '+=', '-=', '*=', '/=']
        for op in operators:
            features[f'op_{op}'] = code_string.count(op)
        features['conditional_count'] = code_string.count('if') + code_string.count('else') + code_string.count('elif')
        features['loop_count'] = code_string.count('for') + code_string.count('while')
        return features

def train_sklearn_model(model, X_train, y_train, X_test, y_test, model_name):
    print(f"Training {model_name}...")
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    print(f"{model_name} - Accuracy: {accuracy:.4f}, Training Time: {training_time:.2f}s")
    return {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'training_time': training_time,
        'predictions': y_pred
    }

def run_ngram_tfidf_svm(train_data, test_data):
    print("Creating n-gram TF-IDF features...")
    vectorizer = TfidfVectorizer(
        max_features=5000,
        ngram_range=(2, 5),
        analyzer='char_wb',
        lowercase=False
    )
    X_train = vectorizer.fit_transform(train_data["code"].astype(str))
    X_test = vectorizer.transform(test_data["code"].astype(str))
    y_train = train_data["EncodedTags"].values
    y_test = test_data["EncodedTags"].values
    model = SVC(kernel='linear', C=1.0, probability=True, random_state=Config.seed)
    return train_sklearn_model(model, X_train, y_train, X_test, y_test, "N-gram TF-IDF SVM")

def run_bow_logistic(train_data, test_data):
    print("Creating Bag of Words features...")
    vectorizer = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        analyzer='word',
        token_pattern=r'\b\w+\b'
    )
    X_train = vectorizer.fit_transform(train_data["code"].astype(str))
    X_test = vectorizer.transform(test_data["code"].astype(str))
    y_train = train_data["EncodedTags"].values
    y_test = test_data["EncodedTags"].values
    model = LogisticRegression(max_iter=1000, random_state=Config.seed, n_jobs=-1)
    return train_sklearn_model(model, X_train, y_train, X_test, y_test, "BoW Logistic Regression")

def run_handcrafted_lgb(train_data, test_data):
    feature_extractor = CodeFeatureExtractor()
    print("Extracting handcrafted features...")
    train_features = []
    test_features = []
    for code in train_data["code"].astype(str):
        features = feature_extractor.extract_features(code)
        train_features.append(list(features.values()))
    for code in test_data["code"].astype(str):
        features = feature_extractor.extract_features(code)
        test_features.append(list(features.values()))
    X_train = np.array(train_features)
    X_test = np.array(test_features)
    y_train = train_data["EncodedTags"].values
    y_test = test_data["EncodedTags"].values
    model = lgb.LGBMClassifier(
        n_estimators=100,
        num_leaves=31,
        learning_rate=0.1,
        random_state=Config.seed,
        n_jobs=-1,
        verbose=-1
    )
    return train_sklearn_model(model, X_train, y_train, X_test, y_test, "Handcrafted LightGBM")

def run_comparison():
    print("Loading datasets...")
    train_data = pd.read_csv(Config.train_path)
    test_data = pd.read_csv(Config.test_path)
    tag_vocab = pd.read_csv(Config.tag_vocab_path)["Tag"].tolist()
    
    label_encoder = LabelEncoder()
    label_encoder.fit(tag_vocab)
    train_data["EncodedTags"] = label_encoder.transform(train_data["language"])
    test_data["EncodedTags"] = label_encoder.transform(test_data["language"])
    
    print(f"Number of classes: {len(tag_vocab)}")
    print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")
    
    model_dispatcher = {
        "ngram_tfidf_svm": run_ngram_tfidf_svm,
        "bow_logistic": run_bow_logistic,
        "handcrafted_lgb": run_handcrafted_lgb,
    }
    
    all_results = {}
    for model_name in Config.models_to_test:
        print(f"\n{'#'*80}")
        print(f"Running: {model_name}")
        print(f"{'#'*80}")
        
        try:
            if model_name in model_dispatcher:
                results = model_dispatcher[model_name](train_data, test_data)
                all_results[model_name] = results
                print(f"✓ {model_name} completed")
            else:
                print(f"✗ {model_name}: Not implemented")
        except Exception as e:
            print(f"✗ Error in {model_name}: {str(e)}")
            all_results[model_name] = {"error": str(e)}
    
    generate_summary(all_results)
    return all_results

def generate_summary(all_results):
    successful_models = {k: v for k, v in all_results.items() if isinstance(v, dict) and 'error' not in v}
    
    if not successful_models:
        print("No successful models")
        return
    
    summary_data = []
    for model_name, results in successful_models.items():
        summary_data.append({
            'Model': model_name,
            'Accuracy': results.get('accuracy', 0),
            'Precision': results.get('precision', 0),
            'Recall': results.get('recall', 0),
            'F1-Score': results.get('f1', 0),
            'Training Time (s)': results.get('training_time', 0),
        })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary = df_summary.sort_values('Accuracy', ascending=False)
    
    os.makedirs(Config.results_dir, exist_ok=True)
    summary_path = os.path.join(Config.results_dir, "model_comparison_summary.csv")
    df_summary.to_csv(summary_path, index=False)
    
    print("\n" + "="*80)
    print("MODEL COMPARISON SUMMARY")
    print("="*80)
    print(df_summary.to_string(index=False))
    
    if len(df_summary) > 0:
        best_model = df_summary.iloc[0]
        print(f"\n🏆 Best Model: {best_model['Model']} (Accuracy: {best_model['Accuracy']:.4f})")

if __name__ == "__main__":
    all_results = run_comparison()
    print(f"\nCompleted: {sum(1 for r in all_results.values() if 'error' not in r)}/{len(Config.models_to_test)} models successfully")